# A Simple App to Convert PDF to Markdown and Send to Azure OpenAI

This notebook demonstrates how to build a simple Python app that reads the content from a PDF converts it to Markdown, and sends it to the Azure OpenAI o3 Reasoning Model API.

## Functions
- `extract_text_from_pdf(pdf_path)`: Extracts text from each page of the PDF and adds a Markdown header for each page. Wraps the detailed text in a code block.
- `send_to_azure_openai(markdown_text, api_endpoint, api_key, deployment_id)`: Sends the provided markdown_text to the Azure OpenAI model. Customize the endpoint, key, and model/deployment ID as needed.
- Execute the workflow to process the input file and send the converted Markdown content to the Azure OpenAI API.

## Usage
Run the script from the command line with your file as an argument:
- The app checks the file extension to choose whether to treat the input as a PDF (using PyPDF2).
- For PDF files, it extracts text page by page and adds Markdown headers. The text is wrapped in a ` ```txt ` block to help retain preformatted text.
- The Markdown-formatted content is then sent to the Azure OpenAI o3 Reasoning Model via a POST request. (Ensure you update the api_endpoint, api_key, and deployment_id accordingly.)
- The app prints the API response to the terminal and also saves the Markdown output to `output.md` for inspection.

## create a virtual environment in your terminal and install the required dependencies to execute the notebook


python -m venv .venv

## Activating the virtual environment on Windows

.venv\Scripts\activate

## Dependencies are in requirements.txt file

Install the required packages using the following command:
```
pip install -r requirements.txt
```

In [36]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [37]:
# Import required libraries
import os
import sys
import json
import requests
from PyPDF2 import PdfReader

### Function to extract text from PDF and convert to Markdown

In [38]:
def extract_text_from_pdf(pdf_path):
    """
    Extracts text from each page of the PDF and adds a Markdown header for each page.
    Wraps the detailed text in a code block.
    """
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PdfReader(f)
        # Loop through pages and add a header for each page
        for i, page in enumerate(reader.pages):
            page_text = page.extract_text()
            if page_text:
                text += f"## Page {i+1}\n\n"
                text += page_text + "\n\n"
    # Optionally wrap in a code block for preserving the exact text format:
    markdown_content = "```txt\n" + text + "```"
    return markdown_content

"Analyze the report data comprehensively and answer the provided questions relating to key takeaways, performance comparisons, and drivers of percentage change using the defined metrics (SOT, AUR, Conversion). Ensure your responses are clear, detailed, and use quantitative reasoning where applicable. ### Input Details: 1. **Key Takeaways** - Highlight three critical takeaways, including strengths and areas for improvement, considering the current report. - Identify three key takeaways when comparing this report to the previous week, noting both strengths and areas for improvement. 2. **Comparative Metrics Analysis** - Determine if the Comp Traffic % increased relative to last week and last year. 3. **Sales and Traffic Insight** - Analyze the relationship between sales lifts and traffic reductions to determine if the lift was driven by SOT or Conversion. 4. **AUR and Sales Mix** - Assess whether high-priced items contributed to a high sales mix, subsequently increasing AUR. 5. **Drivers of Comp % Changes (SOT, AUR, Conversion)** - For each brand in the report, examine which drivers (SOT, AUR, Conversion) significantly contribute to the percentage change when comparing to last year (Comp % to LY). ### Additional Definitions: - **SOT (Sales Over Traffic)** = Overall sales divided by total traffic. - **AUR (Average Unit Retail)** = Average sales price per unit. - **Conversion** = Transactions divided by total traffic. --- # Steps 1. **Extract Key Takeaways** - Analyze the report for high-level trends, areas of success, and opportunities for improvement. Provide three key takeaways for each of the following: - The report itself (current period). - A comparison to the previous week. 2. **Assess Comparative Metrics** - Examine traffic data to determine Comp Traffic % changes: - Compared to last week. - Compared to last year. 3. **Analyze Traffic and Sales Relationship** - If sales increased but traffic decreased: - Assess if the lift is due to higher SOT. - Alternatively, determine if Conversion is the driving factor. 4. **Evaluate AUR Changes** - Check if AUR increases were driven by a higher sales mix of premium (high-priced) items. 5. **Driver Analysis for Percentage Change** - For each brand, identify the key drivers (SOT, AUR, Conversion) contributing to Comp % changes compared to last year (Comp % to LY). - Provide specific reasoning where possible. --- # Output Format The output should be structured as follows: 1. **Key Takeaways (Current Report)** - List three takeaways. - Example: \"[Takeaway 1]\" - Example: \"[Takeaway 2]\" - Example: \"[Takeaway 3]\" 2. **Key Takeaways (Compared to Previous Week)** - List three takeaways. - Example: \"[Takeaway 1]\" - Example: \"[Takeaway 2]\" - Example: \"[Takeaway 3]\" 3. **Comp Traffic % Analysis** - Comparison to Last Week: [Answer if Comp Traffic % increased, with reasoning or data.] - Comparison to Last Year: [Answer if Comp Traffic % increased, with reasoning or data.] 4. **Sales vs. Traffic Analysis** - [Explain whether higher SOT or Conversion caused the sales lift despite traffic decreases.] 5. **AUR and Sales Mix** - [State if higher-priced items contributed to a high sales mix driving up AUR.] 6. **Drivers of Comp % Changes** - [List each brand. Specify which driver(s) (SOT, AUR, or Conversion) are contributing most significantly to Comp % changes compared to last year.] --- # Notes - Use quantitative data from the report wherever possible to support your analysis. - If the report data is missing or ambiguous, structure responses based on logical interpretation of available metrics. - Maintain clarity and conciseness in all sections, avoiding excessive detail unless supporting calculations or reasoning is required."


### Function to send Markdown content to Azure OpenAI

In [39]:
def send_to_azure_openai(markdown_text, api_endpoint, api_key, deployment_id):
    """
    Sends the provided markdown_text to the Azure OpenAI model.
    Customize the endpoint, key, and model/deployment ID as needed.
    """
    # Build the API URL – be sure the API version is supported by your service.
    url = f"{api_endpoint}/openai/deployments/{deployment_id}/chat/completions?api-version=2024-12-01-preview"
    
    print(f"url ", url)
    headers = {
         "Content-Type": "application/json",
         "api-key": api_key,
    }
    # Construct a conversation payload. You can modify the system or user prompts as needed.
    payload = {
      "messages": [
          {"role": "system", "content": "You are a Microsoft Inc corporate financial analysis expert. Analyze the report data comprehensively key takeaways, performance comparisons, and drivers of percentage change. Ensure your responses are clear, detailed, and use quantitative reasoning where applicable. # Notes - Use quantitative data from the report wherever possible to support your analysis. - Maintain clarity and conciseness in all sections, avoiding excessive detail unless supporting calculations or reasoning is required.\n"},
          {"role": "user", "content": markdown_text}
      ],
      #"max_tokens": 1000  # you can adjust based on your model's requirements
}
    response = requests.post(url, headers=headers, data=json.dumps(payload))
    if response.status_code != 200:
         print("Error calling Azure OpenAI API:", response.text)
         return None
    #save_as_pdf(response.text, filename="financial_insights.pdf")
    return response.json()

In [ ]:
def format_financial_report(content, output_file="formatted_financial_report.txt"):
    """
    Format financial report content into readable sections
    filepath: /financial_report.html
    """
    import re
    
    # Function to clean Unicode characters
    def clean_text(text):
        unicode_chars = {
            '\u2010': '-',  # hyphen
            '\u2013': '-',  # en dash
            '\u2014': '--', # em dash
            '\u2019': "'",  # apostrophe
            '\u2003': '  ', # em space
            '\u2500': '-'   # horizontal line
        }
        for char, replacement in unicode_chars.items():
            text = text.replace(char, replacement)
        return text
    
    # Split into major sections
    def format_section(section):
        # Add proper spacing around headers
        if re.match(r'^\d+\.', section.strip()):
            return f"\n{'='*50}\n{section}\n{'='*50}\n"
        return section
    
    # Clean and format content
    content = clean_text(content)
    
    # Split by major sections (Pages and numbered sections)
    sections = content.split('\n\n')
    formatted_sections = []
    
    for section in sections:
        if section.strip():
            formatted = format_section(section)
            # Add bullet points indentation
            if '•' in formatted:
                lines = formatted.split('\n')
                formatted = '\n'.join('    ' + line if '•' in line else line 
                                    for line in lines)
            formatted_sections.append(formatted)
    
    # Join with proper spacing
    formatted_content = '\n\n'.join(formatted_sections)
    
    # Save to file
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(formatted_content)
    
    return formatted_content

### Execute workflow to process the input file and send the converted Markdown content to the Azure OpenAI API

In [41]:
import os
import glob
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(".env")

# Get Azure OpenAI credentials from environment variables
api_key = os.getenv('AZURE_OPENAI_API_KEY')
endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')
deployment_id = os.getenv('AZURE_OPENAI_DEPLOYMENT_ID')
# Use the credentials in your code
#print(f"API Key: {api_key}")
print(f"Endpoint: {endpoint}")
print(f"Deployment ID: {deployment_id}")

directory_path = "./pdfs/"  # Replace with your file path
combined_markdown_text = ""
for file_path in glob.glob(os.path.join(directory_path, "*.pdf")):
        filename, file_extension = os.path.splitext(file_path)
        markdown_text = ""

        # Process PDF files
        if file_extension.lower() == ".pdf":
                print("Processing PDF file...")
                markdown_text = extract_text_from_pdf(file_path)
                combined_markdown_text += markdown_text
        else:
                print("Unsupported file type:", file_extension)
                sys.exit(1)

        # (Optional) Save the converted Markdown to a file for inspection.
        output_filename = os.path.join(directory_path, f"{os.path.basename(filename)}.md")
        with open(output_filename, "w", encoding="utf-8") as f:
                f.write(markdown_text)


print("Sending content to Azure Open AI o3 Reasoning Model API...")
result = send_to_azure_openai(combined_markdown_text, endpoint, api_key, deployment_id)

if result:
        print("Response from Azure OpenAI:")
        financial_insights = result["choices"][0]["message"]["content"]
        #print(json.dumps(result, indent=2))
        # Format and save the report
        formatted_report = format_financial_report(financial_insights)

        # Print first few lines to verify formatting
        print("\nFirst few lines of formatted report:")
        print('\n'.join(formatted_report.split('\n')[:10]))
        
else:
        print("Failed to get a response.")



Endpoint: https://reasoningdemoh1843550720.openai.azure.com
Deployment ID: o3-mini
Processing PDF file...
Processing PDF file...
Processing PDF file...
Processing PDF file...
Processing PDF file...
Processing PDF file...
Sending content to Azure Open AI o3 Reasoning Model API...
url  https://reasoningdemoh1843550720.openai.azure.com/openai/deployments/o3-mini/chat/completions?api-version=2024-12-01-preview
Response from Azure OpenAI:

First few lines of formatted report:
Below is a high-level analysis summarizing the key takeaways from the report, with comparisons over time and the main drivers behind the changes:

-----------------------------  
1. OVERALL BALANCE SHEET TREND

    •  Total Assets have grown substantially over the reporting periods. For example, total assets increased from about 333.8 million at 6/30/2021 to roughly 512.2 million at 6/30/2024 - an increase of over 53% in three years. Such growth appears driven by higher investments in non‑current assets (for instance, 

In [57]:
from enum import Enum
from typing import Union
from pydantic import BaseModel
import json
from openai import AzureOpenAI
import os

# Initialize the Azure OpenAI client
client = AzureOpenAI(
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"), 
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),  
    api_version="2024-12-01-preview"
)

# Define the structure for the formatted output
class GetFormattedOutput(BaseModel):
        introduction: str
        keytakeaways1: str
        keytakeaways2: str
        keytakeaways3: str
        conclusion: str
        delivery_date: str

# Read the financial report text file
file_path = "formatted_financial_report.txt"
try:
        with open(file_path, "r", encoding="utf-8") as f:
                financial_report_content = f.read()
except FileNotFoundError:
        financial_report_content = "File not found. Please ensure formatted_financial_report.txt exists."
except Exception as e:
        financial_report_content = f"Error reading file: {str(e)}"

# Set up tools for function calling using JSON schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_formatted_output",
            "description": "Extract structured information from a financial report",
            "parameters": GetFormattedOutput.model_json_schema()
        }
    }
]

# Set up messages for the API call
messages = [
        {"role": "system", "content": "You are a financial analysis assistant. Extract structured information from the provided financial report following the output format defined."},
        {"role": "user", "content": f"Here is a financial report. Please analyze it and extract the key information according to the specified format:\n\n{financial_report_content}"}
]

# Make the API call to Azure OpenAI
response = client.chat.completions.create(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT_ID", "o3"),
    messages=messages,
    tools=tools,
    tool_choice={"type": "function", "function": {"name": "get_formatted_output"}}
)

# Process and display the response
if response.choices[0].message.tool_calls:
        tool_call = response.choices[0].message.tool_calls[0]
        print(f"Function: {tool_call.function.name}")
        print(f"Arguments: {tool_call.function.arguments}")
        
        # Parse the JSON response
        formatted_output = json.loads(tool_call.function.arguments)
        
        # Create an HTML version of the output
        html_report = f'''
    <html>
    <head>
      <meta http-equiv="Content-Type" content="text/html; charset=windows-1252">
    </head>
    <body>
    <div style="font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif; max-width: 650px; margin: 0 auto; background-color: #ffffff; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden; box-shadow: 0 2px 8px rgba(0,0,0,0.05);">
      <!-- Brand Banner -->
      <div style="background-color: #ffffff; padding: 20px 0; text-align: center; border-bottom: 1px solid #e0e0e0;">
        <img src="https://1.bp.blogspot.com/-UQxhRYz7uDo/W1txbr3hE_I/AAAAAAAABQw/7ry4rLdaLkoeb8pnsfOSOm68ViTfukVCACLcBGAs/s1600/Microsoft-for-desktop.jpg" alt="Microsoft " style="width: 100%; max-width: 100px; height: auto;">
      </div>      

      <!-- Introduction with icon -->
      <div style="padding: 25px 30px; background-color: #f8f9fa; border-bottom: 1px solid #e0e0e0;">
        <table width="100%" cellpadding="0" cellspacing="0" border="0">
          <tr>
        <td width="40" valign="top">
          <div style="width: 30px; height: 30px; background-color: #1e3a8a; border-radius: 50%; text-align: center; line-height: 30px;">
            <span style="color: #ffffff; font-weight: bold; font-size: 16px;">i</span>
          </div>
        </td>
        <td style="padding-left: 15px;">
          <p style="color: #1e3a8a; font-size: 15px; line-height: 1.5; margin: 0; font-weight: 400;">
            {formatted_output['introduction']}
          </p>
        </td>
          </tr>
        </table>
      </div>

      <!-- Key Takeaways Section with placeholders -->
      <div style="padding: 30px; border-bottom: 1px solid #e0e0e0; background: linear-gradient(to right, #ffffff 0%, #f8f9fa 100%);">
        <table width="100%" cellpadding="0" cellspacing="0" border="0">
          <tr>
        <td>
          <h2 style="color: #1e3a8a; font-size: 20px; margin: 0 0 20px; padding-bottom: 12px; border-bottom: 2px solid #1e3a8a; font-weight: 600;">
            KEY TAKEAWAYS
          </h2>
          <!-- Key Takeaway 1 -->
          <table width="100%" cellpadding="0" cellspacing="0" border="0" style="margin-bottom: 20px;">
            <tr>
              <td width="5" style="background-color: #1e3a8a; border-radius: 3px;"></td>
              <td style="padding-left: 15px;">
            <p style="color: #333333; font-size: 15px; line-height: 1.6; margin: 0 0 10px; font-weight: 600;">
              Balance Sheet Analysis
            </p>
            <p style="color: #555555; font-size: 14px; line-height: 1.6; margin: 0;">
              {formatted_output['keytakeaways1']}
            </p>
              </td>
            </tr>
          </table>

          <!-- Key Takeaway 2 -->
          <table width="100%" cellpadding="0" cellspacing="0" border="0" style="margin-bottom: 20px;">
            <tr>
              <td width="5" style="background-color: #1e3a8a; border-radius: 3px;"></td>
              <td style="padding-left: 15px;">
            <p style="color: #333333; font-size: 15px; line-height: 1.6; margin: 0 0 10px; font-weight: 600;">
              Income Statement Analysis
            </p>
            <p style="color: #555555; font-size: 14px; line-height: 1.6; margin: 0;">
              {formatted_output['keytakeaways2']}
            </p>
              </td>
            </tr>
          </table>

          <!-- Key Takeaway 3 -->
          <table width="100%" cellpadding="0" cellspacing="0" border="0">
            <tr>
              <td width="5" style="background-color: #1e3a8a; border-radius: 3px;"></td>
              <td style="padding-left: 15px;">
            <p style="color: #333333; font-size: 15px; line-height: 1.6; margin: 0 0 10px; font-weight: 600;">
              Cash Flow Statement Analysis
            </p>
            <p style="color: #555555; font-size: 14px; line-height: 1.6; margin: 0;">
              {formatted_output['keytakeaways3']}
            </p>
              </td>
            </tr>
          </table>
        </td>
          </tr>
        </table>
      </div>

      <!-- Footer with enhanced styling -->
      <!--div style="background: linear-gradient(135deg, #1e3a8a 0%, #2a4cad 100%); padding: 25px; text-align: center; border-top: 4px solid #4a69dd;"-->
        <table width="100%" cellpadding="0" cellspacing="0" border="0">
          <tr>
        <td align="center">
          <p style="-size: 14px; margin: 0 0 10px; color: #1e3a8a; font-weight: 300; align: left;">
            {formatted_output['conclusion']}
          </p>
          <div style="width: 30px; height: 2px; background-color: rgba(255,255,255,0.3); margin: 15px auto;"></div>
          <p style="color: #1e3a8a; font-size: 12px; margin: 0 0 5px; font-weight: 300; align: center;">
            This report generated for demo purpose with reasoning models.
          </p>
        </td>
          </tr>
        </table>
      <!--/div-->
    </div>
    </body>
    </html>
    '''
        
        # Save the HTML report to a file
        with open("financial_report.html", "w", encoding="utf-8") as html_file:
            html_file.write(html_report)
            
        print("HTML report saved as financial_report.html")

        # Print formatted output
        print("\n--- FORMATTED FINANCIAL REPORT ---")
        print(f"Introduction:\n{formatted_output['introduction']}\n")
        print(f"Key Takeaway 1:\n{formatted_output['keytakeaways1']}\n")
        print(f"Key Takeaway 2:\n{formatted_output['keytakeaways2']}\n")
        print(f"Key Takeaway 3:\n{formatted_output['keytakeaways3']}\n")
        print(f"Conclusion:\n{formatted_output['conclusion']}\n")
        print(f"Delivery Date:\n{formatted_output['delivery_date']}")
else:
        print("No structured output was generated.")

Function: get_formatted_output
Arguments: {
  "introduction": "The report provides a comprehensive analysis of the company’s financial performance over multiple reporting periods. Key areas covered include the expansion of the balance sheet, improvements in revenue and profitability, and strong operating cash flow generation, all of which indicate that the company is successfully financing its growth and delivering enhanced shareholder returns.",
  "keytakeaways1": "Balance Sheet Trend: Total assets increased by over 50% from approximately 333.8 million in 2021 to 512.2 million in 2024, driven primarily by higher investments in non‑current assets such as net PPE. Despite a slight decline in one segment of current assets, overall liquidity is maintained, with an increase in current liabilities and total liabilities being partly offset by robust equity support.",
  "keytakeaways2": "Income Statement & Operating Performance: Revenue grew by about 36% from 198.3 million in 2021 to 270.0 mi

In [ ]:
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
import smtplib
import os

def send_email_with_attachment(sender, recipients, subject, text_file_path, zip_file_path=None):
    # Create the email message
    msg = MIMEMultipart()
    msg['Subject'] = subject
    msg['From'] = sender
    msg['To'] = recipients
    
    # Read the text file content
    with open(text_file_path, 'r', encoding='utf-8') as file:
        file_content = file.read()
    
    # Create HTML content
    html_content = f"""
    <html>
      <body>
        <p>Please find the attached report and a preview below:</p>
        <pre style="background-color: #f5f5f5; padding: 10px; border: 1px solid #ddd;">
        {file_content[:500]}{'...' if len(file_content) > 500 else ''}
        </pre>
        <p>The complete report is attached to this email.</p>
      </body>
    </html>
    """
    
    # Add HTML content to email
    msg.attach(MIMEText(html_content, 'html'))
    
    # Attach the text file
    attachment = MIMEBase('application', 'octet-stream')
    with open(text_file_path, 'rb') as file:
        attachment.set_payload(file.read())
    encoders.encode_base64(attachment)
    attachment.add_header(
        'Content-Disposition', 
        f'attachment; filename="{os.path.basename(text_file_path)}"'
    )
    msg.attach(attachment)
    
    # Attach the ZIP file if provided
    if zip_file_path and os.path.exists(zip_file_path):
        zip_attachment = MIMEBase('application', 'zip')
        with open(zip_file_path, 'rb') as file:
            zip_attachment.set_payload(file.read())
        encoders.encode_base64(zip_attachment)
        zip_attachment.add_header(
            'Content-Disposition', 
            f'attachment; filename="{os.path.basename(zip_file_path)}"'
        )
        msg.attach(zip_attachment)
        print(f"ZIP file attached: {os.path.basename(zip_file_path)}")
    
    # Send the email
    try:
        server = smtplib.SMTP('smtpservername.domain.com', 25, timeout=10)
        server.ehlo()
        server.starttls()
        server.sendmail(sender, recipients.split(","), msg.as_string())
        server.close()
        print(f"Email sent successfully with attachment: {os.path.basename(text_file_path)}")
        if zip_file_path and os.path.exists(zip_file_path):
            print(f"And ZIP attachment: {os.path.basename(zip_file_path)}")
    except Exception as e:
        print(f"Failed to send email: {e}")

# Example usage
sender = "sender1@domain.com"
recipients = "receipient1@domain.com"
subject = "Financial Report Analysis"
text_file_path = "formatted_financial_report.txt"  # Path to text file
zip_file_path = r"./pdfs/FinReport.zip"  # Path to ZIP file

send_email_with_attachment(sender, recipients, subject, text_file_path, zip_file_path)

## Utility function to count tokens per PDF file processing if needed

In [45]:
import fitz  # PyMuPDF
import tiktoken  # Handles counting the tokens

def count_gpt_tokens_from_pdf(pdf_filepath, encoding):
    """
    This function opens a PDF file, extracts the text, converts it to GPT tokens using the specified encoding,
    and counts the number of tokens. This should only be used a rough estimate of the pdf file, the just looks
    at the raw text of each page.

    Args:
        pdf_filepath (str): The path to the PDF file.
        encoding (str):  See tiktoken for supported encodings.

    Returns:
        int: The number of GPT tokens in the PDF text.
    """

    # Encode text to GPT tokens using the given encoding
    encoder = tiktoken.get_encoding(encoding)
    # Open the PDF document
    num_tokens = 0
    page_num = 1
    with fitz.open(pdf_filepath) as doc:
        # Extract text from all pages
        for page in doc:
            text = page.get_text()
            tokens = encoder.encode(text)

            # Count the number of tokens
            page_tokens = len(tokens)
            num_tokens += page_tokens
            print(f"Page {page_num} has {page_tokens} total:{num_tokens}")
            page_num += 1

    return num_tokens

# Example usage
pdf_filepath = "./pdfs/Microsoft Corporation (MSFT) Annual Income Statement - Yahoo Finance.pdf"
encoding = "cl100k_base"
number_of_tokens = count_gpt_tokens_from_pdf(pdf_filepath, encoding)

print(
    f"The PDF file '{pdf_filepath}' has approximately {number_of_tokens} GPT tokens."
)


Page 1 has 1438 total:1438
Page 2 has 148 total:1586
The PDF file './pdfs/Microsoft Corporation (MSFT) Annual Income Statement - Yahoo Finance.pdf' has approximately 1586 GPT tokens.
